In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage

from dotenv import load_dotenv

import os

load_dotenv()
MODEL = os.getenv("MODEL", "gpt-4o-mini")
print(f"{MODEL=}")

MODEL='gpt-4o-mini'


## Creating subagents

In [ ]:
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x**0.5


@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x**2

In [ ]:
# create subagents

subagent_1 = create_agent(
    model=MODEL, tools=[square_root], name="calculate_square_root"
)

subagent_2 = create_agent(model=MODEL, tools=[square], name="calculate_square")

In [15]:
subagent_1.name

'calculate_square_root'

## Calling subagents

In [8]:
@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke(
        {"messages": [HumanMessage(content=f"Calculate the square root of {x}")]}
    )
    return response["messages"][-1].content


@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke(
        {"messages": [HumanMessage(content=f"Calculate the square of {x}")]}
    )
    return response["messages"][-1].content


## Creating the main agent

main_agent = create_agent(
    model=MODEL,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.",
)

## Test

In [9]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [10]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='f6e05824-efc2-4fd7-bf64-940138e9f918'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 118, 'total_tokens': 135, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_94b68f41a1', 'id': 'chatcmpl-EUWpLMhCDA2XkzVH7XF4tC0ZV3wLT', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0fc9f-e98c-7fb1-bb3b-71b45f290dd5-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'call_KuZvYhv8NrE

In [11]:
question = "What is the square root of 456? and square of 50?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [12]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456? and square of 50?', additional_kwargs={}, response_metadata={}, id='1e449dcb-d381-48b4-ab71-524e406bc567'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 124, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_94b68f41a1', 'id': 'chatcmpl-EUWqAhbRzj4UMzq6x4qmaLDa4Xhgm', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0fca0-b747-77b3-8e2d-8759ecacfeec-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id':

In [14]:
response['messages'][1].tool_calls

[{'name': 'call_subagent_1',
  'args': {'x': 456},
  'id': 'call_LyDYxHd2ZB7tJfZU2kfuUPxo',
  'type': 'tool_call'},
 {'name': 'call_subagent_2',
  'args': {'x': 50},
  'id': 'call_0WfeGYngcMozggQShl8hJjQJ',
  'type': 'tool_call'}]